# systematic-execution-ops: results walkthrough

Loads `results/*.json` and the DuckDB store produced by `scripts/run_all.sh` and shows the figures and the headline numbers. Run the pipeline first (`scripts/run_all.sh --skip-download` works on the committed data).

In [ ]:
import json, os
import duckdb, pandas as pd
from IPython.display import Image, display
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
load = lambda n: json.load(open(f'results/{n}', encoding='utf-8')) if os.path.exists(f'results/{n}') else None
def q(sql):
    con = duckdb.connect('data/derived/xops.duckdb', read_only=True)
    try:
        return con.execute(sql).df()
    finally:
        con.close()

## 1. The run: one fault day on a timeline

In [ ]:
run = load('run.json')
days = pd.DataFrame(run['days'])[['date', 'fault_day', 'orders', 'fills', 'dropcopy_fills', 'incidents', 'faults_injected', 'faults_detected', 'checks_warn', 'checks_fail']]
display(days.head(12))
display(Image('results/figures/day.png'))

## 2. Live monitor: fault injection, time to detect, false alerts

In [ ]:
m = load('monitor.json')
print(f"{m['detected']} of {m['injected']} faults caught ({100*m['detection_rate']:.0f}%), median time to detect {m['median_ttd_s']:.0f}s, {m['false_alerts_per_clean_day']:.1f} alerts per clean day")
display(pd.DataFrame(m['by_type']).T)
display(Image('results/figures/monitor.png'))

In [ ]:
q("select rule, severity, count(*) n from alerts group by 1,2 order by n desc")

## 3. End-of-day reconciliation in SQL

In [ ]:
rc = load('recon.json')
print(f"{rc['fills_reconciled']:,} fills reconciled; seeded breaks found {rc['found']}/{rc['seeded']}; unexplained {rc['unexplained']}")
display(pd.DataFrame(rc['breaks_by_type_and_label']))
display(Image('results/figures/recon.png'))

In [ ]:
q("select date, break_type, seeded, symbol, key, detail from recon_breaks order by date, break_type limit 20")

## 4. Corporate actions and calendars against the data

In [ ]:
c = load('corpact.json'); cal = load('calendar_check.json')
print(c['note'])
display(pd.DataFrame(c['as_reported']['by_exchange']).T, pd.DataFrame(c['lse_dividends_in_pounds']['by_exchange']).T)
display(pd.DataFrame({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, list)} for k, v in cal.items()}).T)
display(Image('results/figures/corpact.png'))

## 5. Transaction costs, the broker wheel and the pre-trade model

In [ ]:
t = load('tca.json')
display(pd.DataFrame(t['summary']['by_algo']).T, pd.DataFrame(t['wheel']['brokers']).T, pd.Series(t['wheel']['power']), pd.Series(t['pretrade']))
display(Image('results/figures/tca.png'))

## 6. Portfolio-to-orders and the FIX sessions

In [ ]:
display(Image('results/figures/orders.png'))
q("select date, broker, count(*) orders, sum(n_fills) fills, median(ack_latency_ms) ack_ms, sum(status = 'Filled') filled, sum(status = 'DoneForDay') dfd, sum(status = 'Canceled') canceled, sum(status = 'Rejected') rejected from orders group by 1,2 order by 1,2 limit 12")

## 7. Checks and the fixed-income calendars

In [ ]:
display(Image('results/figures/checks.png'))
fi = load('fi_check.json'); display(pd.DataFrame(fi['futures']).T); print(fi['front_today'])
q("select phase, check_name, status, count(*) n from checks group by 1,2,3 order by 1,2,3")

## 8. Tests

In [ ]:
print(open('results/tests.txt', encoding='utf-8').read() if os.path.exists('results/tests.txt') else 'run the pipeline first')